# Content Refresh Opportunity Scoring

**Author:** Khanh Nguyen Quoc  
**Lane:** Refresh / Content Opportunity Scoring  
**Repository:** [KhanhGiauTen/flyrankAI](https://github.com/KhanhGiauTen/flyrankAI)

## Abstract

This study asks which visible pages a content team should inspect first when review capacity is limited. It uses the public-safe FlyRank starter snapshot of 30,000 pseudonymized content items from 32 clients, with trailing-90-day search, content, and engagement signals. A transparent position-adjusted CTR rule and three classifiers are compared on the same six-client holdout using Precision@20/50, average precision, and ROC-AUC. The constrained random forest tied the simple rule at Precision@20 (**0.85**) and Precision@50 (**0.84**) while improving average precision only modestly (**0.709 vs 0.695**), so complexity did not earn operational preference. The practical output is a ranked, reason-coded review queue that keeps a human editor in control and is explicitly limited to directional decision support.

## 1. Question

**Which visible pages should a content editor review first for a possible title, snippet, or search-intent mismatch?** The unit of analysis is one pseudonymized content item in a trailing-90-day snapshot. The output is a ranked queue, not an automatic edit. A false positive wastes review capacity and may prompt an unnecessary change; a false negative leaves a real opportunity unattended.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists()), Path.cwd())
data_path = repo_root / 'data/raw/content_refresh_anonymized.csv'
output_dir = repo_root / 'work/outputs'
df = pd.read_csv(data_path)
question_contract = {
    'decision': 'which pages enter the next human review batch',
    'unit': 'one pseudonymized content item',
    'output': 'ranked queue with one reason code and action label',
    'review_capacity': 50,
}
question_contract


{'decision': 'which pages enter the next human review batch',
 'unit': 'one pseudonymized content item',
 'output': 'ranked queue with one reason code and action label',
 'review_capacity': 50}

## 2. Data

The analysis uses `data/raw/content_refresh_anonymized.csv`, the public-safe starter release: 30,000 rows × 44 columns, 32 pseudonymized clients, and one row per content item. Search, content, and engagement measurements cover a trailing 90-day window; the file does not expose a calendar endpoint, so none is invented here. Rates are stored as percentages (`0.76` means 0.76%), and average position `0` means unavailable. Client/content IDs are context only. Direct label fields (`trend_direction`, `trend_pct`), their recent/previous comparison inputs, provider/model names, and IDs are excluded from model features.

In [2]:
data_audit = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'pseudonymized_clients': df.client_id.nunique(),
    'unique_content_items': df.content_id.nunique(),
    'valid_position_rows': int(df.avg_position.gt(0).sum()),
    'pages_with_500plus_impressions': int(df.impressions_90d.ge(500).sum()),
})
assert data_audit['rows'] == data_audit['unique_content_items']
assert not df['content_id'].astype(str).str.contains(r'https?://', regex=True).any()
data_audit.to_frame('observed_value')


,observed_value
rows,30000
columns,44
pseudonymized_clients,32
unique_content_items,30000
valid_position_rows,28795
pages_with_500plus_impressions,16726


## 3. Methodology

The teaching proxy is `trend_direction == 'down'`, meaning observed impressions fell more than 20% from the previous 30-day window to the latest one. It is a current-window proxy, not a future label and not proof that a refresh will help. The frozen baseline scores pages with at least 500 impressions and positions 1–50 using `log1p(impressions) × max(position-band median CTR − observed CTR, 0)`, assigning `low_ctr_visible_page` and a human-review action. Logistic regression, a depth-limited decision tree, and a constrained random forest use safe snapshot features with median/missingness-aware preprocessing. Six clients (20%) are held out entirely, and the baseline medians are estimated on training clients before both rule and models are scored on the identical test rows.

In [3]:
baseline_receipt = json.loads((output_dir / 'baseline_metrics.json').read_text(encoding='utf-8'))
model_receipt = json.loads((output_dir / 'model_metrics.json').read_text(encoding='utf-8'))
assert model_receipt['seed'] == 42
assert model_receipt['split'] == 'client_holdout'
assert model_receipt['test_clients'] == 6
method_audit = pd.Series({
    'seed': model_receipt['seed'],
    'train_rows': model_receipt['train_rows'],
    'test_rows': model_receipt['test_rows'],
    'held_out_clients': model_receipt['test_clients'],
    'excluded_direct_answer_fields': len(model_receipt['excluded_direct_answer_fields']),
})
method_audit.to_frame('value')


,value
seed,42
train_rows,12759
test_rows,3578
held_out_clients,6
excluded_direct_answer_fields,13


## 4. Results (same split, same metrics)

The frozen Week-4 rule is already strong at the review cutoffs. The random forest ties it at Precision@20 and Precision@50 and improves whole-ranking average precision by about 0.014, while ROC-AUC remains close. This is a useful negative result: the nonlinear model does not deliver enough top-of-queue gain to justify replacing the simpler operational rule. The rule therefore remains the recommended decision aid, with the random forest retained as a research comparator.

In [4]:
results = pd.DataFrame(model_receipt['comparison']).set_index('method')
display(results.style.format('{:.3f}').highlight_max(axis=0, color='#d9ead3'))
rule = results.loc['week4_rule']
forest = results.loc['random_forest']
headline = pd.Series({
    'rule_precision_at_50': rule.precision_at_50,
    'forest_precision_at_50': forest.precision_at_50,
    'rule_average_precision': rule.average_precision,
    'forest_average_precision': forest.average_precision,
    'average_precision_gain': forest.average_precision - rule.average_precision,
    'recommended_operational_method': 'week4_rule',
})
headline.to_frame('value')


,precision_at_20,precision_at_50,average_precision,roc_auc
method,,,,
base_rate,0.650,0.580,0.611,0.500
week4_rule,0.850,0.840,0.695,0.618
logistic_regression,0.800,0.760,0.709,0.617
decision_tree,0.650,0.820,0.678,0.614
random_forest,0.850,0.840,0.709,0.620


,value
rule_precision_at_50,0.84
forest_precision_at_50,0.84
rule_average_precision,0.694665
forest_average_precision,0.709119
average_precision_gain,0.014454
recommended_operational_method,week4_rule


## 5. Limitations and honest framing

This snapshot cannot establish causality, reveal Google ranking factors, or demonstrate future predictive skill. The proxy and many features overlap the same 90-day snapshot, so the experiment is a teaching comparison of ranking methods, not a sealed forecast. Aggregate CTR can hide query, device, geography, brand intent, and SERP-feature differences. Missing keyword and word-count data follow content type, and six held-out clients are still a limited generalization test. A production follow-up needs a feature window that ends before a future target window, repeated grouped/time validation, and human review of every recommendation.

In [5]:
limitations = pd.DataFrame([
    ('Target', 'Current-window teaching proxy; not a future outcome'),
    ('Causality', 'No intervention or causal design'),
    ('Aggregation', 'Query/device/SERP mix is hidden inside page-level totals'),
    ('Missingness', 'Systematic by content type, not random'),
    ('Generalization', 'One fixed six-client holdout'),
], columns=['risk', 'boundary'])
limitations


,risk,boundary
0,Target,Current-window teaching proxy; not a future ou...
1,Causality,No intervention or causal design
2,Aggregation,Query/device/SERP mix is hidden inside page-le...
3,Missingness,"Systematic by content type, not random"
4,Generalization,One fixed six-client holdout


## 6. Ranked recommendations

1. **Start with high-volume page-one CTR gaps.** Review title, snippet, and intent fit; do not edit automatically.
2. **Check context before acting.** Query mix, brand/navigation intent, device mix, and SERP features can make a position-band median misleading.
3. **Use one reason code per queue item.** `low_ctr_visible_page` makes the recommendation auditable and easy to challenge.
4. **Keep the rule as the default.** The random forest did not beat it at the top-20 or top-50 cutoffs.
5. **Measure after review.** A later, non-overlapping window should determine whether the page improved; this snapshot alone cannot attribute recovery to an edit.

In [6]:
queue = pd.read_csv(output_dir / 'baseline_action_score.csv')
top10 = queue.head(10)[[
    'rank', 'content_id', 'impressions_90d', 'avg_position', 'ctr',
    'expected_ctr', 'reason_code', 'action_label'
]]
top10


,rank,content_id,impressions_90d,avg_position,ctr,expected_ctr,reason_code,action_label
0,1,content_c8e9d6ab9013,208678,9.7,0.00,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
1,2,content_453722754fea,140079,7.6,0.01,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
2,3,content_39881853ef0c,112434,7.2,0.01,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
3,4,content_c84a0ab98e90,223271,7.8,0.03,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
4,5,content_0919dd345d80,119217,7.0,0.02,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
5,6,content_d274ac4158ef,65138,6.8,0.01,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
6,7,content_e5f459e737b7,56363,5.9,0.01,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
7,8,content_c1fe78bc4e37,134055,7.5,0.03,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
8,9,content_339b357d04c7,46879,3.7,0.01,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"
9,10,content_65114d89496d,72631,6.5,0.02,0.24,low_ctr_visible_page,"review title, snippet, and search-intent fit"


## 7. Reproducibility and paper artifacts

From a fresh clone, install `requirements.txt`, run `scripts/run_all.py`, then execute `w04_baseline_score.ipynb`, `w05_model.ipynb`, and this notebook in order. Random state 42 fixes the client selection and estimators. `work/outputs/baseline_metrics.json`, `work/outputs/model_metrics.json`, and `work/outputs/capstone_summary.json` are small receipts that preserve the reported numbers. The deployed paper links back to these notebooks and the public repository.

In [7]:
summary = {
    'title': 'Content Refresh Opportunity Scoring',
    'author': 'Khanh Nguyen Quoc',
    'lane': 'Refresh / Content Opportunity Scoring',
    'dataset_rows': int(len(df)),
    'pseudonymized_clients': int(df.client_id.nunique()),
    'split': model_receipt['split'],
    'test_clients': model_receipt['test_clients'],
    'test_proxy_base_rate': model_receipt['test_proxy_base_rate'],
    'recommended_method': 'week4_rule',
    'week4_rule': results.loc['week4_rule'].to_dict(),
    'random_forest': results.loc['random_forest'].to_dict(),
    'top10_public_safe_ids': top10['content_id'].tolist(),
    'seed': 42,
}
(output_dir / 'capstone_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f"Wrote {output_dir / 'capstone_summary.json'}")
summary


Wrote C:\Users\Acer\source\repos\FlyrankAI\flyrankAI\work\outputs\capstone_summary.json


{'title': 'Content Refresh Opportunity Scoring',
 'author': 'Khanh Nguyen Quoc',
 'lane': 'Refresh / Content Opportunity Scoring',
 'dataset_rows': 30000,
 'pseudonymized_clients': 32,
 'split': 'client_holdout',
 'test_clients': 6,
 'test_proxy_base_rate': 0.6106763555058692,
 'recommended_method': 'week4_rule',
 'week4_rule': {'precision_at_20': 0.85,
  'precision_at_50': 0.84,
  'average_precision': 0.6946649508,
  'roc_auc': 0.6184040503},
 'random_forest': {'precision_at_20': 0.85,
  'precision_at_50': 0.84,
  'average_precision': 0.7091193942,
  'roc_auc': 0.6204878594},
 'top10_public_safe_ids': ['content_c8e9d6ab9013',
  'content_453722754fea',
  'content_39881853ef0c',
  'content_c84a0ab98e90',
  'content_0919dd345d80',
  'content_d274ac4158ef',
  'content_e5f459e737b7',
  'content_c1fe78bc4e37',
  'content_339b357d04c7',
  'content_65114d89496d'],
 'seed': 42}

## 8. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). The project uses only the public-safe pseudonymized starter snapshot and credits FlyRank as the data source.

## 9. Communication cuts (ML-12)

### Five-minute demo outline

1. The decision: which visible pages deserve the next 50 human reviews?
2. The data and safety boundary: 30,000 pseudonymized pages; no raw URLs, clients, or queries.
3. The baseline: a position-adjusted CTR gap weighted by visibility.
4. The honest comparison: six-client holdout; rule and random forest both reach 0.84 Precision@50.
5. The conclusion: keep the simpler rule, require human review, and validate on a future window next.

### Social-post cut

I built a content-refresh opportunity queue on 30,000 public-safe FlyRank page records. On a six-client holdout, a transparent position-adjusted CTR rule tied a constrained random forest at 0.84 Precision@50; the model only modestly improved average precision. The useful lesson was not “more ML wins” — it was that a readable baseline can remain the better operational choice when complexity does not improve the decision cutoff.

### Employer-facing summary

I built and validated a reason-coded content opportunity ranking system on 30,000 pseudonymized search/content records. I compared a transparent baseline with logistic regression, a decision tree, and a random forest using a client-group holdout and top-K metrics. The random forest tied the rule at Precision@50, so I recommended the simpler method and documented leakage risks, subgroup errors, reproducibility, and limits.

## Self-check

- [x] Every section is filled with executed evidence
- [x] The notebook runs top to bottom with no errors
- [x] No client names, domains, URLs from data, or private queries appear
- [x] Claims are observed, measured, directional, and decision-support only
- [x] The paper contains all nine required sections
- [x] Baseline and models use the same held-out rows and metrics
- [x] Reproducibility receipts and fixed seed are recorded
- [x] Five-minute demo, social-post cut, and employer summary are included